# AWS Glue Job Documentation Notebook

---

## **1. Glue Job Overview**

**Job Name:** `glue_snowpark_job`

**IAM Role:** `gluesnowparkrole`

**Script Type:** Python Shell (or Glue Python)

**Purpose:**

* Fetch files from a GitHub repository folder.
* Upload files to a specified S3 bucket.



In [1]:


## **2. Python Script for Glue Job**

import os
import requests
import boto3
from io import BytesIO

# GitHub repository details
GITHUB_REPO = "https://api.github.com/repos/{owner}/{repo}/contents/{folder}"
OWNER = "deacademygit"
REPO = "project-data"
FOLDER = "snowpark-data"  # Folder in GitHub repo

HEADERS = {"Accept": "application/vnd.github.v3+json"}

# S3 Bucket details
BUCKET_NAME = "YOUR_BUCKET_NAME"  # Replace with your bucket

# Initialize S3 client
s3 = boto3.client("s3")

def fetch_github_files():
    """Fetch file details from GitHub folder."""
    url = GITHUB_REPO.format(owner=OWNER, repo=REPO, folder=FOLDER)
    response = requests.get(url, headers=HEADERS)
    if response.status_code == 200:
        return response.json()  # List of file details
    else:
        print("Failed to fetch files:", response.text)
        return []

def upload_to_s3(file_url, file_name):
    """Download file from GitHub and upload it to S3."""
    response = requests.get(file_url)
    if response.status_code == 200:
        s3.upload_fileobj(BytesIO(response.content), BUCKET_NAME, file_name)
        print(f"Uploaded {file_name} to S3 bucket {BUCKET_NAME}")
    else:
        print(f"Failed to download {file_name}: {response.text}")

def main():
    files = fetch_github_files()
    for file in files:
        if file["type"] == "file":  # Ignore directories
            file_url = file["download_url"]
            file_name = file["name"]
            upload_to_s3(file_url, file_name)

if __name__ == "__main__":
    main()





ModuleNotFoundError: No module named 'boto3'


## **3. Steps to Run Glue Job**

1. Create a Glue job in AWS Console:

   * Name: `glue_snowpark_job`
   * IAM Role: `gluesnowparkrole`
   * Type: Python Shell (or Glue Python)
   * Script Location: Upload this script to S3 or paste it in Glue editor.

2. Replace `YOUR_BUCKET_NAME` with your actual S3 bucket.

3. Save the job.

4. Run the job:

   * Go to **AWS Glue Jobs** console.
   * Select `glue_snowpark_job`.
   * Click **Run job**.

5. Check logs in **CloudWatch Logs** to confirm:

   * Files fetched from GitHub.
   * Files uploaded to S3.

---

## **4. Expected Output**

* Files from the GitHub folder `snowpark-data` should appear in your S3 bucket.
* Logs will print a line for every uploaded file: `Uploaded <filename> to S3 bucket <BUCKET_NAME>`.

